# IDX-Trade for Humans

Tujuan notebook ini bukan menjalankan seluruh research system. Ini adalah peta untuk memahami repo dengan mata sendiri: branch aktif, modul model historis, dan bagaimana melihat source tanpa checkout branch lain.

**Safety:** notebook ini hanya membaca Git metadata/source. Tidak membuka outcome vault, tidak fit model, dan tidak menulis artifact.

In [ ]:
from pathlib import Path
import subprocess

REPO = Path.cwd()
while REPO != REPO.parent and not (REPO / ".git").exists():
    REPO = REPO.parent

print("repo:", REPO)

## 1. Branch kamu sekarang

Kalau `src/idx_trade` hanya berisi `ranking_v4_3_*`, itu normal untuk V4-3 lineage. Older research code tetap ada di historical branches.

In [ ]:
def git(*args):
    return subprocess.run(
        ["git", *args],
        cwd=REPO,
        check=True,
        text=True,
        capture_output=True,
    ).stdout.strip()

print("current branch:", git("branch", "--show-current"))
print("HEAD:", git("rev-parse", "--short", "HEAD"))
print("\nRelevant remote branches:")
branches = git("branch", "-r").splitlines()
for b in branches:
    if any(x in b.lower() for x in ("ranking-v2", "ranking-v3", "ohlcv-o2", "ranking-v4-3")):
        print(b.strip())

## 2. Source map

Historical model code tidak harus ada di working tree branch saat ini.

- Clean V2: `origin/research/idx-ranking-v2-spec-v1`
- V3-B Structure-Lite: historical V2/V3 development tree
- O2: `origin/research/idx-ranking-ohlcv-o2-final-refit-v1`
- V4-3: current branch family

In [ ]:
SOURCE_FILES = {
    "V2 model": (
        "origin/research/idx-ranking-v2-spec-v1",
        "src/idx_trade/research_v2_models.py",
    ),
    "V2 features": (
        "origin/research/idx-ranking-v2-spec-v1",
        "src/idx_trade/research_v2_features.py",
    ),
    "V3-B": (
        "origin/research/idx-ranking-v2-spec-v1",
        "src/idx_trade/ranking_v3_structure_lite.py",
    ),
    "O2 final refit": (
        "origin/research/idx-ranking-ohlcv-o2-final-refit-v1",
        "src/idx_trade/ohlcv_o2_final_refit.py",
    ),
    "V4-3 model eval": (
        "HEAD",
        "src/idx_trade/ranking_v4_3_model_eval.py",
    ),
}

for name, (ref, path) in SOURCE_FILES.items():
    proc = subprocess.run(
        ["git", "show", f"{ref}:{path}"],
        cwd=REPO,
        text=True,
        capture_output=True,
    )
    print(f"{name:18s}", "FOUND" if proc.returncode == 0 else "NOT IN LOCAL REFS")

Kalau historical refs belum ada, jalankan manual di terminal:

```bash
git fetch origin --prune
```

Lalu rerun cell di atas.

In [ ]:
# Preview 40 baris pertama source V2 tanpa checkout branch.
ref, path = SOURCE_FILES["V2 model"]
src = git("show", f"{ref}:{path}")
print("\n".join(src.splitlines()[:40]))

## 3. Mental model repo

Pikirkan repo seperti ini:

`branch eksperimen = snapshot code + contract untuk satu research lane`

Bukan:

`setiap model baru mengedit satu file model.py yang sama`

Karena itu V2, V3-B, O2, dan V4-3 bisa hidup bersamaan di Git history walaupun working tree saat ini hanya menampilkan salah satunya.